In [2]:
import torch

In [3]:
X_data = torch.tensor([[1,2,3],[1,2,3]]) # n x p (n samples, p features)
w = torch.tensor([[0.1],[0.2],[0.3]], requires_grad=True)   # p x 1 (p features, 1 output)
print(X_data.shape)
print(w.shape)

torch.Size([2, 3])
torch.Size([3, 1])


In [7]:
# starting with 1 d case just to know how it works
a = torch.tensor(2.0,requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(4.0, requires_grad=True)

# y = a + b 
# z = y * x  

y = a + b 
z = x * y
print(y.grad_fn)
print(z.grad_fn)
print(a.grad_fn)

None


## Initial Experiment (using p=n=1 for simplicity)

In [ ]:
true_linear_kernel_1 = torch.tensor(50) # so  the true value of sigma is (5 or -5)
true_sigma = 5

x_i_1 = torch.tensor(1)
x_j_1 = torch.tensor(2)

sigma_1 = torch.tensor(0.5, requires_grad=True)
sigma_squared_1 = sigma_1 ** 2
linear_kernel_1 = sigma_squared_1 * x_i_1 * x_j_1
print(linear_kernel_1.requires_grad) # why is this True? becaus sigma_1 requires grad and in order to compute its grad we need to know how linear_kernel_1 depends on sigma_1; PyTorch must track gradients for any tensor that depends (directly or indirectly) on a tensor that requires gradients


True


In [ ]:
# initialization of sigma_1 
sigma_1_pred = torch.tensor(0.5, requires_grad=True)

# forward pass 
sigma_squared_1_pred = sigma_1_pred ** 2
linear_kernel_1_pred = sigma_squared_1_pred * x_i_1 * x_j_1
print(linear_kernel_1_pred.requires_grad) # why is this True? becaus sigma_1 requires grad and in order to compute its grad we need to know how linear_kernel_1 depends on sigma_1; PyTorch must track gradients for any tensor that depends (directly or indirectly) on a tensor that requires gradients
print(linear_kernel_1_pred)

# backward pass
loss_1 = (linear_kernel_1_pred - true_linear_kernel_1) ** 2
print(loss_1)
loss_1.backward()
print(sigma_1_pred.grad)  # d loss_1 / d sigma_1;;;; the gradient always points in the direction of the steepest increase,->> in our case gradient is negative, so this means if we were to increase sigma_1_pred the loss will decrease (simple terms to minimize the loss go in the opposite direction of the gradient)

# act on: update sigma_1_pred using gradient descent
learning_rate = 0.01
with torch.no_grad():  # we don't want to track this operation in the computation graph
    sigma_1_pred -= learning_rate * sigma_1_pred.grad
    sigma_1_pred.grad.zero_()  # reset the gradient to zero for the next iteration; because if we do not do this the old gradient will be accumulated with the new gradient in the next backward pass
    print(sigma_1_pred)

# let us now put this in a loop to see how sigma_1_pred converges to the true value

True
tensor(0.5000, grad_fn=<MulBackward0>)
tensor(2450.2500, grad_fn=<PowBackward0>)
tensor(-198.)
tensor(2.4800, requires_grad=True)


In [39]:
epochs = 100
learning_rate = 0.001
sigma_1_pred = torch.tensor(0.5, requires_grad=True)
for epoch in range(epochs):
    # forward pass
    sigma_squared_1_pred = sigma_1_pred ** 2
    linear_kernel_1_pred = sigma_squared_1_pred * x_i_1 * x_j_1

    # compute loss
    loss_1 = (linear_kernel_1_pred - true_linear_kernel_1) ** 2

    # backward pass
    loss_1.backward()

    # update sigma_1_pred
    with torch.no_grad():
        sigma_1_pred -= learning_rate * sigma_1_pred.grad
        sigma_1_pred.grad.zero_()

    print(f"Epoch {epoch}: sigma_1_pred = {sigma_1_pred.item()}, loss = {loss_1.item()}")

Epoch 0: sigma_1_pred = 0.6980000138282776, loss = 2450.25
Epoch 1: sigma_1_pred = 0.9717589616775513, loss = 2403.5087890625
Epoch 2: sigma_1_pred = 1.3457802534103394, loss = 2314.703857421875
Epoch 3: sigma_1_pred = 1.8450943231582642, loss = 2150.895751953125
Epoch 4: sigma_1_pred = 2.4826297760009766, loss = 1865.484375
Epoch 5: sigma_1_pred = 3.2308566570281982, loss = 1419.262451171875
Epoch 6: sigma_1_pred = 3.983597993850708, loss = 848.15673828125
Epoch 7: sigma_1_pred = 4.565582275390625, loss = 333.4967956542969
Epoch 8: sigma_1_pred = 4.869135856628418, loss = 69.07134246826172
Epoch 9: sigma_1_pred = 4.969752788543701, loss = 6.6720476150512695
Epoch 10: sigma_1_pred = 4.993731498718262, loss = 0.36374780535697937
Epoch 11: sigma_1_pred = 4.99873685836792, loss = 0.01569756306707859
Epoch 12: sigma_1_pred = 4.999747276306152, loss = 0.0006381143466569483
Epoch 13: sigma_1_pred = 4.9999494552612305, loss = 2.5547706172801554e-05
Epoch 14: sigma_1_pred = 4.999989986419678, 

### Experiment 2

#### True Values 

In [51]:
n = 2 
p = 5 
torch.manual_seed(0)
X_data = torch.randn(n, p)  # n x p
true_sigmas = torch.randn(p)
true_linear_kernel_matrix =  torch.ones(n,n) # shape n x n ; k_i_j = sum over sigmas of sigma_k^2 * x_i_k * x_j_k where k goes from 1 to p, and  i and j go from 1 to n

for i in range(n): 
    for j in range(n):
        k_i_j = 0 
        for k in range(p): 
            k_i_j += true_sigmas[k]**2 * X_data[i,k] * X_data[j,k]
        true_linear_kernel_matrix[i,j] = k_i_j
print(f"true_linear_kernel_matrix:\n {true_linear_kernel_matrix}")
print(f"true_sigmas:\n {true_sigmas}")

true_linear_kernel_matrix:
 tensor([[ 6.0731, -2.1044],
        [-2.1044,  2.0304]])
true_sigmas:
 tensor([-0.5966,  0.1820, -0.8567,  1.1006, -1.0712])


In [ ]:
# initialization of sigmas_pred
sigmas_pred = torch.randn(p,1, requires_grad=True)
print(f"Initial sigmas_pred:\n {sigmas_pred}")

# training loop
epochs = 10000
learning_rate = 0.001

for epoch in range(epochs): 
    # forward pass: (compute THE WHOLE linear kernel matrix, and then compute loss)#NOTE: is there another way ? 
    linear_kernel_matrix_pred = torch.ones(n,n) # shape n x n, these will be now overwrritten in the nested loops
    for i in range(n): 
        for j in range(n): 
            k_i_j_pred = 0 
            for k in range(p): 
                k_i_j_pred += sigmas_pred[k]**2 * X_data[i,k] * X_data[j,k]
            linear_kernel_matrix_pred[i,j] = k_i_j_pred
    # compute loss
    loss = torch.sum((linear_kernel_matrix_pred - true_linear_kernel_matrix) ** 2 ) # MSE loss over all entries in the kernel matrix (element-wise difference squared and summed)
    # backward pass
    loss.backward()
    # update sigmas_pred
    with torch.no_grad():
        sigmas_pred -= learning_rate * sigmas_pred.grad
        sigmas_pred.grad.zero_()
    if epoch % 10 == 0:
        # print the whole array of sigmas_pred
        print(f"Epoch {epoch}: sigmas_pred = {sigmas_pred.view(-1)}, loss = {loss.item()}")
print(f"Final sigmas_pred:\n {sigmas_pred.view(-1)}")

### Experiment 3: with the loss = -MLL 

In [6]:
# now my goal is to maximize the marginal log likelihood, so I will define my loss = - MLL (because we care to maximize MLL which is equivalent to minimizing - MLL)
import torch 
n = 2 
p = 5 
torch.manual_seed(0)
X_data = torch.randn(n, p)  # n x p
y_data = torch.randn(n,1)  # n x 1 

#NOTE maybe try to see which values of sigmas give me N(0,I) since this is the data distribution I have assumed for y_data, and I will try to maximize p(y|X, sigmas) by changing sigmas

#NOTE: Is doing maximizing p(y|X, sigmas) equivalent to minimizing the difference between the predicted kernel matrix and I ?  (since y ~ N(0,I) i.e. like we were doing previously)


# intialization of the kernel matrix 
# kernel_matrix = torch.randn(n,n)

sigma_square_of_normal_noise = torch.tensor(1)  #  by sigma_square_of_normal_noise I mean the variance of the normal noise added to the outputs (y= f(x) + N(0, sigma^2))

sigmas_pred = torch.randn(p,1, requires_grad=True)
learning_rate = 0.001

epochs = 10000
for epoch in range(epochs): 
    linear_kernel_pred = torch.ones(n,n) # these we do not care about their value they are just placeholder for now #TODO: investigate if the gradients will track chnages from one to k_i_j or not ( I think no )
    # instead of this loop;;;; I will do matrix multiplication 
    for i in range(n):
        for j in range(n): 
            k_i_j = 0
            for k in range(p):
                k_i_j += sigmas_pred[k]**2 * X_data[i,k] * X_data[j,k]
            linear_kernel_pred[i,j] = k_i_j
    # Computation of the loss 
    K = linear_kernel_pred + sigma_square_of_normal_noise * torch.eye(n) 
    K_inv = torch.linalg.inv(K)
    mu = torch.zeros(n,1)
    sub = y_data - mu
    num = torch.exp(-0.5* (sub.T @ K_inv @ sub))
    den = torch.sqrt((2*torch.pi)**n * torch.det(K))
    MLL = num/den
    loss = - MLL
    loss.backward()

    # update sigmas_pred
    with torch.no_grad():
        sigmas_pred -= learning_rate * sigmas_pred.grad # p x 1 
        sigmas_pred.grad.zero_()

    if epoch % 10 == 0: 
        print(f"Probability became {MLL}")



Probability became tensor([[0.0380]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0380]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0380]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0381]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0382]], grad_fn=<DivBackward0>)
Probability became tensor([[0.0382]], grad_fn=<DivBackward0>)
Probabil